# Day 3 · 3교시 [실습 보조] ML 파이프라인의 아래층 — `03_ml_pipeline`

## 실습 목표

전처리→학습→평가→반복을 직접 돌리고, 무엇보다 **AI가 조용히 저지르는 함정(데이터 누수)**을
점수로 실측한다. ML 코드는 문법 오류가 없어도 **방법론이 틀리면** 가짜 점수를 낸다 —
그래서 AI가 짠 ML 코드는 특히 사람이 검증해야 한다(교안 3.8).

| 순서 | 내용 | 교안 |
|------|------|------|
| 1 | 데이터 로드·탐색 | 3.3 |
| 2 | 전처리(분할 먼저→스케일) | 3.4 |
| 3 | 학습(baseline) | 3.5 |
| 4 | 평가(혼동행렬) | 3.6 |
| 5 | 실험 반복(스윕) | 3.7 |
| 6 | **함정: 데이터 누수** 실측 | 3.8 |

> 전부 오프라인(sklearn 내장·합성 데이터), 결정적(random_state 고정). 의존성: `scikit-learn·numpy`.

In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

X, y, names = *load_wine(return_X_y=True), load_wine().target_names
print("데이터:", X.shape, "| 클래스:", list(names))
print("클래스 분포:", dict(zip(names, np.bincount(y))))

데이터: (178, 13) | 클래스: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]
클래스 분포: {np.str_('class_0'): np.int64(59), np.str_('class_1'): np.int64(71), np.str_('class_2'): np.int64(48)}


## 2. 전처리 — 분할 먼저, 스케일은 그 다음 (순서가 생명)

교안 3.4·3.8의 철칙: **train/test 분할을 먼저** 하고, 스케일러는 **train 에만 fit** 한 뒤
test 에는 transform 만. 순서를 뒤집으면 test 정보가 train 에 새어 든다(6절에서 실측).

In [2]:
from sklearn.preprocessing import StandardScaler

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scaler = StandardScaler().fit(Xtr)          # ← train 에만 fit
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)               # test 는 transform 만 (fit 아님)
print("분할:", Xtr.shape[0], "train /", Xte.shape[0], "test  — 스케일러는 train 에만 fit")

분할: 124 train / 54 test  — 스케일러는 train 에만 fit


## 3. 학습 — baseline 부터 (3.5)

복잡한 모델 전에 단순한 기준선. 로지스틱 회귀로 baseline 정확도를 잡는다.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ytr)
print(f"baseline train acc: {accuracy_score(ytr, clf.predict(Xtr_s)):.3f}")
print(f"baseline test  acc: {accuracy_score(yte, clf.predict(Xte_s)):.3f}   ← 일반화 성능")

baseline train acc: 1.000
baseline test  acc: 0.981   ← 일반화 성능


## 4. 평가 — 혼동행렬 (3.6)

정확도 하나로는 부족하다. 혼동행렬·정밀도·재현율로 *어느 클래스를 어떻게 틀리는지* 본다.

In [4]:
from sklearn.metrics import confusion_matrix, classification_report

pred = clf.predict(Xte_s)
print("혼동행렬 (행=실제, 열=예측):")
cm = confusion_matrix(yte, pred)
print("        " + "  ".join(f"{n[:6]:>6}" for n in names))
for i, row in enumerate(cm):
    print(f"{names[i][:6]:>6}  " + "  ".join(f"{v:>6}" for v in row))
print("\n" + classification_report(yte, pred, target_names=names, digits=2))

혼동행렬 (행=실제, 열=예측):
        class_  class_  class_
class_      18       0       0
class_       1      20       0
class_       0       0      15

              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        18
     class_1       1.00      0.95      0.98        21
     class_2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54



## 5. 실험 반복을 위임 (3.7)

사람은 *어떤 축을 실험할지*만 정하고, 반복은 손발에 맡긴다. RandomForest 의 나무 수를
스윕해 비교 표를 만든다.

In [5]:
from sklearn.ensemble import RandomForestClassifier

print(f"{'n_estimators':>12} | {'test acc':>8}")
for n in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42).fit(Xtr_s, ytr)
    print(f"{n:>12} | {accuracy_score(yte, rf.predict(Xte_s)):>8.3f}")
print("→ 사람은 '무엇을 실험할지'(가설), 반복은 에이전트에게(3.1)")

n_estimators | test acc
          10 |    0.963
          50 |    1.000


         100 |    1.000


         200 |    1.000
→ 사람은 '무엇을 실험할지'(가설), 반복은 에이전트에게(3.1)


## 6. 함정 실측 — 데이터 누수(leakage)가 점수를 부풀린다

교안 3.8의 핵심. 노이즈 특성이 많은 합성 데이터에서, **특성 선택을 분할 전(전체 데이터)** 에
하면 test 정답이 선택에 새어 들어 **가짜로 높은 점수**가 나온다. 올바른 버전(train 에서만
선택)과 비교한다 — 실행은 둘 다 잘 되지만(문법 오류 없음) 점수가 다르다.

In [6]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif

# 정보 있는 특성 5개 + 노이즈 195개 = 누수가 크게 드러나는 세팅
Xr, yr = make_classification(n_samples=300, n_features=200, n_informative=5,
                             n_redundant=0, random_state=0)

def eval_pipeline(leaky):
    if leaky:  # ✗ 누수: 전체 데이터로 특성 선택(test 정답을 봄) → 그 다음 분할
        Xsel = SelectKBest(f_classif, k=20).fit_transform(Xr, yr)
        a, b, c, d = train_test_split(Xsel, yr, test_size=0.3, random_state=1)
    else:      # ✓ 올바름: 분할 먼저 → train 에서만 특성 선택
        a, b, c, d = train_test_split(Xr, yr, test_size=0.3, random_state=1)
        sel = SelectKBest(f_classif, k=20).fit(a, c)
        a, b = sel.transform(a), sel.transform(b)
    m = LogisticRegression(max_iter=1000).fit(a, c)
    return accuracy_score(d, m.predict(b))

leaky_acc = eval_pipeline(leaky=True)
ok_acc = eval_pipeline(leaky=False)
print(f"누수 있는 버전(✗): test acc = {leaky_acc:.3f}   ← 부풀려진 가짜 점수")
print(f"올바른 버전(✓):    test acc = {ok_acc:.3f}   ← 정직한 점수")
print(f"\n격차 {leaky_acc - ok_acc:+.3f} — 문법은 둘 다 정상. '방법론 오류'라 실행돼도 틀린다(3.8)")

누수 있는 버전(✗): test acc = 0.833   ← 부풀려진 가짜 점수
올바른 버전(✓):    test acc = 0.778   ← 정직한 점수

격차 +0.056 — 문법은 둘 다 정상. '방법론 오류'라 실행돼도 틀린다(3.8)


## 실습 정리

- 전처리(**분할 먼저→스케일**)→baseline→평가(혼동행렬)→스윕을 직접 돌렸다.
- **데이터 누수**를 점수로 실측: 전체 데이터로 특성 선택하면 test 점수가 부풀려진다 —
  실행은 잘 되지만 틀린 결과. AI가 짠 ML 코드가 특히 위험한 이유.
- 원칙: **반복 손발은 에이전트, 가설·검증은 사람**. 정확도 아닌 **일반화**로 판단하고,
  train/test 격차·누수·잘못된 지표를 사람이 지킨다(3.8·3.9).